#### Inverse Problem

In [1]:
%matplotlib qt

import mindmap as mm
import mne
import numpy as np
from pathlib import Path

config = mm.io.read_config("../configurations/spike.yml")
mm_dir = Path(config["MM_DIR"])

## Data

In [17]:
sub     = "sub-pat10"
forward = mm_dir/sub/"fwd"/f"{sub}_surf-fwd.fif"
gft     = mm_dir/sub/"cst"/f"{sub}_cst-template.npy"
eeg     = mm_dir/sub/"eeg"/f"{sub}_ave.fif"

# parameters sLORETA
lam_loreta = 0.001
loose      = 0
weight     = 0.8

# parameters CSET
lam_cset      = 0.1
rising_phase  = 0.5
initial_guess = "adjoint"

pyxu_kwargs = dict(
    folder = "solverOutput/",
    exist_ok = True,
    writeback_rate = 50,
    verbosity = 1,
    show_progress = False,
    log_var = "x",
)

# plotting parameters
plot_kwargs = dict(
    subject = sub, 
    hemi = "both", 
    surface = "white",
    smoothing_steps = 10,
)

# load
eeg       = mne.read_evokeds(eeg)[0]
forward   = mne.read_forward_solution(forward)
gft       = np.load(gft, allow_pickle=True)
noise_cov = mne.make_ad_hoc_cov(eeg.info, std={"eeg":1})

Reading /home/admin/work/data/eeg_epi_spikeerp_data_1.0.0/derivatives/mindmap/sub-pat10/eeg/sub-pat10_ave.fif ...
    Found the data of interest:
        t =       0.00 ...     996.00 ms (No comment)
        0 CTF compensation matrices available
        nave = 1 - aspect type = 100
No projector specified for this dataset. Please consider the method self.add_proj.
No baseline correction applied
Reading forward solution from /home/admin/work/data/eeg_epi_spikeerp_data_1.0.0/derivatives/mindmap/sub-pat10/fwd/sub-pat10_surf-fwd.fif...
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
    Desired named matrix (kind = 3523) not available
    Read EEG forward solution (6537 sources, 204 channels, free orientations)
    Source spaces transform

## Compute Inverse with sLORETA

In [20]:
solver  = mm.inverse.InverseSolver("sLORETA", eeg, forward, noise_cov)
sloreta = solver.solve(lam_loreta, loose, weight)

EEG channel type selected for re-referencing
Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.
Computing inverse operator with 204 channels.
    204 out of 204 channels remain after picking
Selected 204 channels
Creating the depth weighting matrix...
    204 EEG channels
    limit = 6538/6537 = 2.868567
    scale = 480675 exp = 0.8
    Picked elements from a free-orientation depth-weighting prior into the fixed-orientation one
    Average patch normals will be employed in the rotation to the local surface coordinates....
    Converting to surface-based source orientations...
    [done]
Whitening the forward solution.
    Created an SSP operator (subspace dimension = 1)
Computing rank from covariance with rank=None
    Using tolerance 0.00045 (2.2e-16 eps * 204 dim * 1e+10  max singular value)
    Estimated rank (eeg): 203
    EEG: rank 203 computed from 204

In [21]:
sloreta.plot(clim = {"kind":"value", "lims":[0, sloreta.data.max()/2, sloreta.data.max()]}, **plot_kwargs)

For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


## Compute Inverse with CSET

In [18]:
solver = mm.inverse.InverseSolver("CSET", eeg, forward, gft)
recon  = solver.solve(lam_cset, rising_phase=rising_phase, initial_guess=initial_guess, pyxu_kwargs=pyxu_kwargs)

data = np.abs(recon)

vertices = [
    forward["src"][0]["vertno"],
    forward["src"][1]["vertno"],
]

cset = mne.SourceEstimate(data, vertices, tmin=0, tstep=0, subject=sub)

In [19]:
cset.plot(clim = {"kind":"value", "lims":[0, cset.data.max()/2, cset.data.max()]}, **plot_kwargs)

For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`
For automatic theme detection, "darkdetect" has to be installed! You can install it with `pip install darkdetect`


## Save results

In [25]:
mm_dir        = Path(config["MM_DIR"])
sloreta_fname = mm_dir/sub/"inv"/f"{sub}_sloreta-{lam_loreta}"
cset_fname    = mm_dir/sub/"inv"/f"{sub}_cset-{lam_cset}"

sloreta.save(sloreta_fname, overwrite=True)
cset.save(cset_fname, overwrite=True)

Writing STC to disk...
[done]
Writing STC to disk...
[done]
